# Step 3 — Fill Structural Coverage Gaps

**Pipeline:** Myocarditis structural mimicry reassessment (spike vs. cardiac autoantigens)
**This step:** the first GPU-bound step in this pipeline. Genuinely needs Colab's T4.

**Reads:** `structure_sources.json` (from Step 2) — only touches proteins/regions flagged
`needs_fallback: true`. Anything Step 2 already found experimentally is left alone.
**Produces:** `filled_structures.json` — a record of every fallback structure generated, its
source (AlphaFold DB or ESMFold), per-region confidence, and file paths; plus the structure files
themselves.

### Fallback order (locked from the methodology — do not reorder)

1. **AlphaFold DB first.** Free, pre-computed, EMBL-EBI/DeepMind hosted, keyed on UniProt
   accession. Every predicted region source here is flagged `source: alphafold_db`.
2. **ESMFold only where AlphaFold DB coverage is also incomplete.** Run **locally inside this
   notebook via HuggingFace `transformers`**, not the public ESM Atlas API — that hosted API
   (`api.esmatlas.com`) currently has a known, unresolved SSL certificate problem reported on the
   official facebookresearch/esm GitHub repo, and depending on it here would repeat the exact kind
   of fragile-dependency mistake this pipeline was built to avoid (see Step 1's ElliPro-over-Epitopia
   decision for the same reasoning applied earlier in this project). Every predicted region here is
   flagged `source: esmfold_local`.

Every region filled by either method is flagged `predicted: true` and is **never treated as a
substitute** for the experimentally-resolved conformational diversity that carries this study's
central claim (that distinction lives in spike, which by design should need no fallback here at
all — see the check in the first code cell below).

ESMFold2 was considered and explicitly not used, per the earlier methodology review: no demonstrated
accuracy improvement on disordered/flexible regions, the property that actually matters for this
study, versus a much less proven, days-old release.

## Environment check

Confirm a GPU runtime is actually attached before doing anything else — ESMFold needs it, and folding on CPU would be impractically slow. Runtime -> Change runtime type -> T4 GPU, if this fails.

In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        "No GPU detected. Go to Runtime -> Change runtime type -> Hardware "
        "accelerator -> T4 GPU, then re-run this cell before proceeding."
    )
print(result.stdout)

In [ ]:
import importlib, subprocess, sys

REQUIRED = ["requests", "torch", "transformers", "accelerate"]
for pkg in REQUIRED:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import requests, torch, transformers
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)

## Mount Drive and load `structure_sources.json` from Step 2

In [ ]:
import json, os

MOUNT_DRIVE = True
BASE_DIR = "/content"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/myocarditis_pipeline"

STEP1_DIR = os.path.join(BASE_DIR, "step1")
STEP2_DIR = os.path.join(BASE_DIR, "step2")
STEP3_DIR = os.path.join(BASE_DIR, "step3")
os.makedirs(STEP3_DIR, exist_ok=True)

panel_path = os.path.join(STEP1_DIR, "panel.json")
sources_path = os.path.join(STEP2_DIR, "structure_sources.json")

for p in (panel_path, sources_path):
    if not os.path.exists(p):
        raise FileNotFoundError(f"{p} not found. Run Steps 1 and 2 first.")

with open(panel_path) as f:
    panel = json.load(f)
with open(sources_path) as f:
    structure_sources = json.load(f)

print("Loaded panel.json and structure_sources.json.\n")

# Sanity check: spike should need no fallback by design. If it does, that is
# worth knowing explicitly rather than silently folding spike regions, since
# the study's central claim rests on experimentally-resolved spike states only.
spike_states_missing = structure_sources["spike"]["states_missing"]
if spike_states_missing:
    print(f"NOTE: spike is missing experimental coverage for: {spike_states_missing}")
    print("Per the locked methodology, missing spike conformational states are recorded as")
    print("absent, NOT filled by prediction. Predicted spike structures are out of scope —")
    print("this study's central claim rests on experimentally-resolved states only.\n")

gap_summary = {
    symbol: entry for symbol, entry in structure_sources["cardiac_antigens"].items()
    if entry["needs_fallback"]
}

print(f"{len(gap_summary)} of {len(structure_sources['cardiac_antigens'])} cardiac antigens flagged needs_fallback:")
for symbol, entry in gap_summary.items():
    print(f"  {symbol}: {entry['coverage_pct']}% experimental coverage "
          f"({entry['covered_length_aa']}/{entry['target_length_aa']} aa)")

## Fallback tier 1 — AlphaFold DB

Queried by UniProt accession via the public API (`alphafold.ebi.ac.uk/api/prediction/{accession}`).
Note: AFDB fragments proteins longer than 2700 residues into overlapping 1400-residue pieces
(e.g. titin-style `-F1`, `-F2` naming) — handled explicitly below rather than assuming one clean
file per accession, since MYH6 at ~1939 aa could plausibly fall into fragmented territory depending
on AFDB's current per-protein length handling.

In [ ]:
ALPHAFOLD_API_BASE = "https://alphafold.ebi.ac.uk/api/prediction"
REQUEST_TIMEOUT = 30

def query_alphafold_db(accession: str) -> list:
    """Returns a list of prediction entries for this accession (usually 1, more if fragmented)."""
    url = f"{ALPHAFOLD_API_BASE}/{accession}"
    resp = requests.get(url, timeout=REQUEST_TIMEOUT, headers={"Accept": "application/json"})
    if resp.status_code == 404:
        return []
    resp.raise_for_status()
    return resp.json()


def download_alphafold_structure(entry: dict, dest_dir: str, fmt: str = "cif") -> str:
    url_key = {"pdb": "pdbUrl", "cif": "cifUrl"}[fmt]
    url = entry[url_key]
    filename = url.split("/")[-1]
    dest_path = os.path.join(dest_dir, filename)
    resp = requests.get(url, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    with open(dest_path, "wb") as f:
        f.write(resp.content)
    return dest_path


AFDB_DIR = os.path.join(STEP3_DIR, "alphafold_db")
os.makedirs(AFDB_DIR, exist_ok=True)

afdb_results = {}

for symbol, gap_entry in gap_summary.items():
    accession = panel["entries"][symbol]["accession"]
    print(f"\n{symbol} ({accession}) — querying AlphaFold DB...")

    entries = query_alphafold_db(accession)
    if not entries:
        print(f"  No AlphaFold DB entry found for {accession}. Will need ESMFold fallback.")
        afdb_results[symbol] = {"found": False, "fragments": []}
        continue

    print(f"  {len(entries)} fragment(s)/entry(ies) found.")
    fragments = []
    for e in entries:
        path = download_alphafold_structure(e, AFDB_DIR, fmt="cif")
        frag_info = {
            "entry_id": e.get("entryId"),
            "sequence_start": e.get("sequenceStart"),
            "sequence_end": e.get("sequenceEnd"),
            "mean_plddt": e.get("globalMetricValue"),
            "fraction_plddt_very_low": e.get("fractionPlddtVeryLow"),
            "fraction_plddt_confident": e.get("fractionPlddtConfident"),
            "model_created_date": e.get("modelCreatedDate"),
            "file_path": path,
        }
        fragments.append(frag_info)
        print(f"    {frag_info['entry_id']}  residues {frag_info['sequence_start']}-{frag_info['sequence_end']}  "
              f"mean pLDDT={frag_info['mean_plddt']}")

    afdb_results[symbol] = {"found": True, "fragments": fragments}
    time.sleep(0.3) if "time" in dir() else None

print("\nAlphaFold DB fallback pass complete.")

In [ ]:
import time  # ensure available for the polite-delay call above and below

# Determine actual residual coverage after AFDB: does any fragment set fully
# cover the target sequence, or is there still a gap for ESMFold to fill?

def compute_residual_gap(symbol: str, target_length: int, afdb_entry: dict) -> dict:
    if not afdb_entry["found"]:
        return {"fully_covered": False, "covered_ranges": [], "gap_ranges": [(1, target_length)]}

    covered_ranges = sorted(
        (f["sequence_start"], f["sequence_end"]) for f in afdb_entry["fragments"]
        if f["sequence_start"] is not None and f["sequence_end"] is not None
    )

    # merge overlapping/adjacent ranges
    merged = []
    for start, end in covered_ranges:
        if merged and start <= merged[-1][1] + 1:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))

    # find gaps against 1..target_length
    gaps = []
    cursor = 1
    for start, end in merged:
        if start > cursor:
            gaps.append((cursor, start - 1))
        cursor = max(cursor, end + 1)
    if cursor <= target_length:
        gaps.append((cursor, target_length))

    return {"fully_covered": len(gaps) == 0, "covered_ranges": merged, "gap_ranges": gaps}


residual_gaps = {}
for symbol, gap_entry in gap_summary.items():
    target_length = gap_entry["target_length_aa"]
    residual = compute_residual_gap(symbol, target_length, afdb_results[symbol])
    residual_gaps[symbol] = residual
    status = "fully covered by AlphaFold DB" if residual["fully_covered"] else f"gaps remain: {residual['gap_ranges']}"
    print(f"{symbol}: {status}")

## Fallback tier 2 — ESMFold (local, via HuggingFace)

Only runs for residual gap regions that AlphaFold DB did not cover. Loaded locally through
`transformers`, not the public ESM Atlas API, for the reliability reasons noted above.

The model is ~2.7GB and will take a few minutes to download on first run. Folding is done per gap
region (not necessarily the full protein) to stay within the T4's 16GB VRAM — long single-pass folds
of multi-thousand-residue proteins are not feasible on this hardware, which is expected and is why
this fallback operates on gap *regions*, not full sequences, wherever AlphaFold DB already covered
part of the protein.

In [ ]:
ESMFOLD_MAX_RESIDUES_PER_CALL = 400  # conservative T4-safe chunk size; adjust down if you hit OOM

any_esmfold_needed = any(not r["fully_covered"] for r in residual_gaps.values())

if any_esmfold_needed:
    print("Loading ESMFold locally via HuggingFace transformers (first run downloads ~2.7GB)...")
    from transformers import AutoTokenizer, EsmForProteinFolding
    import torch

    esmfold_tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
    esmfold_model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
    esmfold_model = esmfold_model.cuda()
    esmfold_model.esm = esmfold_model.esm.half()  # reduce memory footprint on T4
    esmfold_model.trunk.set_chunk_size(64)
    print("ESMFold loaded and on GPU.")
else:
    print("No residual gaps after AlphaFold DB — ESMFold is not needed for this run.")
    esmfold_model = None

In [ ]:
def fold_with_esmfold(sequence: str) -> tuple:
    """Returns (pdb_string, mean_plddt) for a single sequence via local ESMFold."""
    with torch.no_grad():
        output = esmfold_model.infer_pdb(sequence)
    # Extract mean pLDDT from the B-factor column of the returned PDB text
    plddts = []
    for line in output.splitlines():
        if line.startswith("ATOM"):
            try:
                plddts.append(float(line[60:66].strip()))
            except ValueError:
                pass
    mean_plddt = sum(plddts) / len(plddts) if plddts else None
    return output, mean_plddt


ESMFOLD_DIR = os.path.join(STEP3_DIR, "esmfold")
os.makedirs(ESMFOLD_DIR, exist_ok=True)

esmfold_results = {}

if esmfold_model is not None:
    for symbol, gap_entry in gap_summary.items():
        residual = residual_gaps[symbol]
        if residual["fully_covered"]:
            continue

        full_seq = panel["entries"][symbol]["sequence"]
        symbol_fragments = []

        for gap_start, gap_end in residual["gap_ranges"]:
            gap_len = gap_end - gap_start + 1
            print(f"\n{symbol}: folding residues {gap_start}-{gap_end} ({gap_len} aa)...")

            if gap_len > ESMFOLD_MAX_RESIDUES_PER_CALL:
                print(f"  WARNING: region exceeds {ESMFOLD_MAX_RESIDUES_PER_CALL} aa chunk size; "
                      f"splitting further. Domain boundaries were not used here — this is a length-based "
                      f"split only. For publication-quality regions, consider folding by actual domain "
                      f"boundary instead of a fixed residue count.")
                # simple sequential sub-chunking as a safety net
                sub_start = gap_start
                while sub_start <= gap_end:
                    sub_end = min(sub_start + ESMFOLD_MAX_RESIDUES_PER_CALL - 1, gap_end)
                    sub_seq = full_seq[sub_start - 1:sub_end]
                    pdb_str, mean_plddt = fold_with_esmfold(sub_seq)
                    fname = f"{symbol}_{sub_start}-{sub_end}_esmfold.pdb"
                    fpath = os.path.join(ESMFOLD_DIR, fname)
                    with open(fpath, "w") as f:
                        f.write(pdb_str)
                    symbol_fragments.append({
                        "region_start": sub_start, "region_end": sub_end,
                        "mean_plddt": mean_plddt, "file_path": fpath,
                    })
                    print(f"    {sub_start}-{sub_end}: mean pLDDT={mean_plddt:.1f}" if mean_plddt else f"    {sub_start}-{sub_end}: folded")
                    sub_start = sub_end + 1
            else:
                sub_seq = full_seq[gap_start - 1:gap_end]
                pdb_str, mean_plddt = fold_with_esmfold(sub_seq)
                fname = f"{symbol}_{gap_start}-{gap_end}_esmfold.pdb"
                fpath = os.path.join(ESMFOLD_DIR, fname)
                with open(fpath, "w") as f:
                    f.write(pdb_str)
                symbol_fragments.append({
                    "region_start": gap_start, "region_end": gap_end,
                    "mean_plddt": mean_plddt, "file_path": fpath,
                })
                print(f"  mean pLDDT={mean_plddt:.1f}" if mean_plddt else "  folded (no pLDDT parsed)")

        esmfold_results[symbol] = symbol_fragments

    print("\nESMFold fallback pass complete.")
else:
    print("Skipped — no residual gaps required ESMFold.")

## Write `filled_structures.json`

Full audit trail: every predicted region generated, its source (AlphaFold DB vs. ESMFold), confidence, and file path. Every entry here is flagged `predicted: true` for Step 4 to carry forward.

In [ ]:
from datetime import datetime, timezone

filled_structures = {
    "pipeline_step": 3,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "generated_in": "Google Colab (GPU runtime)",
    "source_structure_sources": sources_path,
    "spike_fallback_used": False,  # by design — see the note printed earlier in this notebook
    "cardiac_antigens": {},
}

for symbol in gap_summary:
    afdb_entry = afdb_results[symbol]
    residual = residual_gaps[symbol]

    regions = []

    # AlphaFold DB fragments, each flagged with source + predicted=True
    for frag in afdb_entry.get("fragments", []):
        regions.append({
            "source": "alphafold_db",
            "predicted": True,
            "region_start": frag["sequence_start"],
            "region_end": frag["sequence_end"],
            "mean_plddt": frag["mean_plddt"],
            "fraction_plddt_very_low": frag["fraction_plddt_very_low"],
            "file_path": frag["file_path"],
        })

    # ESMFold regions, same flagging
    for frag in esmfold_results.get(symbol, []):
        regions.append({
            "source": "esmfold_local",
            "predicted": True,
            "region_start": frag["region_start"],
            "region_end": frag["region_end"],
            "mean_plddt": frag["mean_plddt"],
            "file_path": frag["file_path"],
        })

    filled_structures["cardiac_antigens"][symbol] = {
        "target_length_aa": gap_summary[symbol]["target_length_aa"],
        "experimental_coverage_pct": gap_summary[symbol]["coverage_pct"],
        "fully_covered_by_alphafold_db": residual["fully_covered"],
        "regions": regions,
    }

# Cardiac antigens that did NOT need fallback are recorded too, for completeness,
# with an empty regions list and predicted=False implied.
for symbol, entry in structure_sources["cardiac_antigens"].items():
    if symbol not in filled_structures["cardiac_antigens"]:
        filled_structures["cardiac_antigens"][symbol] = {
            "target_length_aa": entry["target_length_aa"],
            "experimental_coverage_pct": entry["coverage_pct"],
            "fully_covered_by_alphafold_db": None,
            "regions": [],
            "note": "No fallback needed — fully covered experimentally in Step 2.",
        }

output_path = os.path.join(STEP3_DIR, "filled_structures.json")
with open(output_path, "w") as f:
    json.dump(filled_structures, f, indent=2)

print("filled_structures.json written to:", output_path)
print("\nSummary:")
for symbol, entry in filled_structures["cardiac_antigens"].items():
    n_regions = len(entry["regions"])
    sources = set(r["source"] for r in entry["regions"])
    if n_regions:
        print(f"  {symbol}: {n_regions} predicted region(s), source(s)={sources}")
    else:
        print(f"  {symbol}: no prediction needed")

---

### Before moving to Step 4

Check the mean pLDDT values printed above for every predicted region, low-confidence regions
(roughly pLDDT < 50) are predicted but structurally unreliable, and this is worth knowing before
those regions feed into Step 4's structural alignment. The methods document already commits to
disclosing predicted-vs-experimental status per region; this notebook's output is what makes that
disclosure possible with real numbers rather than a general caveat.

If `esmfold_results` shows any region split into multiple GPU-safe sub-chunks rather than folded
as one continuous piece, consider whether that split crossed a genuine domain boundary or just an
arbitrary residue count — arbitrary splits can introduce artificial discontinuities at the cut
point that a domain-aware split would avoid. This is flagged in the fold loop above but not
automatically corrected.